# QAM16 vs QAM64 raw amplitude and phase crossing analysis

This notebook compares QAM16 and QAM64 at SNR values `-4, -2, 0, and 2 dB` using all signals available at each SNR.

Important rules:

- No random sample selection.
- No normalization.
- No mean subtraction.
- All signals belonging to a modulation and SNR are used.
- Raw amplitude is computed as `A(t) = sqrt(I(t)^2 + Q(t)^2)`.
- Raw phase is computed as `phase(t) = atan2(Q(t), I(t))` in radians.
- For each SNR, QAM16 and QAM64 share the same exact raw min-to-max range.
- That shared range is divided into 20 equal bins.
- For each bin/threshold value, the notebook counts how many time samples cross that value.

Main goal:

For amplitude, this directly checks how often QAM64 reaches high-amplitude/outer-power regions compared with QAM16.

In [ ]:
# CELL 1: Setup
import pickle
import shutil
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

DATA_PATH = Path("/kaggle/input/datasets/gustavopolicarpo/rml201610a-dict/RML2016.10a_dict.dat")

EXP_DIR = Path("/kaggle/working/qam16_qam64_raw_crossing_analysis")
FIG_DIR = EXP_DIR / "figures"
RESULT_DIR = EXP_DIR / "results"

for d in [FIG_DIR, RESULT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Data path: {DATA_PATH}")
print(f"Figures  : {FIG_DIR}")
print(f"Results  : {RESULT_DIR}")

In [ ]:
# CELL 2: Configuration and helper functions
TARGET_MODS = ["QAM16", "QAM64"]
TARGET_SNRS = [-4, -2, 0, 2]
NUM_BINS = 20

COLORS = {
    "QAM16": "#1f77b4",
    "QAM64": "#ff7f0e",
}


def amplitude_from_iq(X):
    """Raw amplitude A(t)=sqrt(I(t)^2+Q(t)^2). X shape: (N,2,128)."""
    X = np.asarray(X, dtype=np.float32)
    return np.sqrt(X[:, 0, :] ** 2 + X[:, 1, :] ** 2)


def phase_from_iq(X):
    """Raw wrapped phase phase(t)=atan2(Q(t),I(t)) in radians. X shape: (N,2,128)."""
    X = np.asarray(X, dtype=np.float32)
    return np.arctan2(X[:, 1, :], X[:, 0, :])


def exact_bins(values, n_bins=20):
    """Create n_bins equal bins from exact min to exact max of raw values."""
    values = np.asarray(values, dtype=np.float64).reshape(-1)
    vmin = float(np.min(values))
    vmax = float(np.max(values))
    if np.isclose(vmin, vmax):
        pad = 1e-6 if vmin == 0 else abs(vmin) * 1e-6
        vmin -= pad
        vmax += pad
    return np.linspace(vmin, vmax, n_bins + 1), vmin, vmax


def crossing_counts(values, thresholds, direction=">="):
    """Count raw time samples crossing each threshold."""
    values = np.asarray(values).reshape(-1)
    rows = []
    total = int(values.size)
    for threshold in thresholds:
        if direction == ">=":
            count = int(np.sum(values >= threshold))
        elif direction == "<=":
            count = int(np.sum(values <= threshold))
        else:
            raise ValueError("direction must be >= or <=")
        rows.append({
            "threshold": float(threshold),
            "crossing_count": count,
            "total_time_samples": total,
            "crossing_fraction": float(count / total),
        })
    return rows


def safe_snr_name(snr):
    return f"snr_{snr:+03d}db".replace("+", "plus_").replace("-", "minus_")

print("Configuration ready")
print(f"Target modulations: {TARGET_MODS}")
print(f"Target SNRs       : {TARGET_SNRS}")
print(f"Shared bins/SNR   : {NUM_BINS}")

In [ ]:
# CELL 3: Load all signals and compute raw amplitude/phase
assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}"

with open(DATA_PATH, "rb") as f:
    data = pickle.load(f, encoding="latin1")

missing = [(m, s) for m in TARGET_MODS for s in TARGET_SNRS if (m, s) not in data]
assert not missing, f"Missing keys in dataset: {missing}"

amp_raw = {}
phase_raw = {}
summary_rows = []

for snr in TARGET_SNRS:
    for mod in TARGET_MODS:
        X = np.asarray(data[(mod, snr)], dtype=np.float32)
        assert X.ndim == 3 and X.shape[1:] == (2, 128), f"Unexpected shape for {(mod, snr)}: {X.shape}"

        A = amplitude_from_iq(X)
        P = phase_from_iq(X)
        amp_raw[(mod, snr)] = A
        phase_raw[(mod, snr)] = P

        summary_rows.append({
            "modulation": mod,
            "snr_db": snr,
            "num_signals": int(X.shape[0]),
            "time_samples_per_signal": int(X.shape[-1]),
            "total_time_samples": int(A.size),
            "raw_amplitude_min": float(A.min()),
            "raw_amplitude_max": float(A.max()),
            "raw_amplitude_mean": float(A.mean()),
            "raw_amplitude_p95": float(np.percentile(A, 95)),
            "raw_amplitude_p99": float(np.percentile(A, 99)),
            "raw_phase_min_rad": float(P.min()),
            "raw_phase_max_rad": float(P.max()),
            "raw_phase_mean_rad": float(P.mean()),
        })

summary_df = pd.DataFrame(summary_rows)
summary_csv = RESULT_DIR / "raw_amplitude_phase_summary_all_signals.csv"
summary_df.to_csv(summary_csv, index=False)

print("Loaded all signals and computed raw amplitude/phase")
display(summary_df.round(4))
print(f"Saved: {summary_csv}")

In [ ]:
# CELL 4: All-signal raw histograms and threshold-crossing counts
# For each SNR, QAM16 and QAM64 share exact combined min-to-max bins.
# Threshold crossing uses the bin-left values. For amplitude this means A(t) >= threshold.

amp_hist_rows = []
phase_hist_rows = []
amp_cross_rows = []
phase_cross_rows = []
outer_rows = []

for snr in TARGET_SNRS:
    # ---------------- Amplitude: shared 20-bin histogram + crossing counts ----------------
    combined_A = np.concatenate([amp_raw[(mod, snr)].reshape(-1) for mod in TARGET_MODS])
    amp_bins, amp_min, amp_max = exact_bins(combined_A, NUM_BINS)
    amp_centers = 0.5 * (amp_bins[:-1] + amp_bins[1:])
    amp_thresholds = amp_bins[:-1]  # 20 threshold values, one per bin-left edge

    fig, ax = plt.subplots(figsize=(9, 5.5))
    for mod in TARGET_MODS:
        values = amp_raw[(mod, snr)].reshape(-1)
        counts, _ = np.histogram(values, bins=amp_bins)

        ax.step(amp_centers, counts, where="mid", color=COLORS[mod], linewidth=2.4, label=mod)
        ax.scatter(amp_centers, counts, color=COLORS[mod], s=18)

        for bin_id, (left, right, center, count) in enumerate(zip(amp_bins[:-1], amp_bins[1:], amp_centers, counts), start=1):
            amp_hist_rows.append({
                "snr_db": snr,
                "modulation": mod,
                "bin_id": bin_id,
                "bin_left": float(left),
                "bin_right": float(right),
                "bin_center": float(center),
                "count": int(count),
            })

        for row in crossing_counts(values, amp_thresholds, direction=">="):
            row.update({"snr_db": snr, "modulation": mod, "feature": "raw_amplitude", "direction": ">="})
            amp_cross_rows.append(row)

        highest_bin_count = int(counts[-1])
        total = int(values.size)
        outer_rows.append({
            "snr_db": snr,
            "modulation": mod,
            "shared_amplitude_min": float(amp_min),
            "shared_amplitude_max": float(amp_max),
            "highest_bin_left_threshold": float(amp_bins[-2]),
            "highest_bin_right": float(amp_bins[-1]),
            "highest_bin_crossing_count": highest_bin_count,
            "total_time_samples": total,
            "highest_bin_crossing_fraction": float(highest_bin_count / total),
        })

    ax.axvspan(amp_bins[-2], amp_bins[-1], color="red", alpha=0.08, label="highest amplitude bin")
    ax.set_title(f"All-signal raw amplitude histogram, exact 20 bins, SNR = {snr} dB")
    ax.set_xlabel("Raw amplitude A(t) = sqrt(I(t)^2 + Q(t)^2)")
    ax.set_ylabel("Number of time samples")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    out_path = FIG_DIR / f"{safe_snr_name(snr)}_all_signals_raw_amplitude_exact20bins.png"
    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")

    fig, ax = plt.subplots(figsize=(9, 5.5))
    for mod in TARGET_MODS:
        df = pd.DataFrame([r for r in amp_cross_rows if r["snr_db"] == snr and r["modulation"] == mod])
        ax.plot(df["threshold"], df["crossing_count"], color=COLORS[mod], linewidth=2.4, marker="o", label=mod)

    ax.axvline(amp_bins[-2], color="red", linestyle="--", linewidth=1.2, label="highest-bin threshold")
    ax.set_title(f"Raw amplitude threshold crossings, SNR = {snr} dB")
    ax.set_xlabel("Raw amplitude threshold value")
    ax.set_ylabel("Number of time samples with A(t) >= threshold")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    out_path = FIG_DIR / f"{safe_snr_name(snr)}_raw_amplitude_crossing_counts.png"
    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")

    # ---------------- Phase: shared 20-bin histogram + crossing counts ----------------
    combined_P = np.concatenate([phase_raw[(mod, snr)].reshape(-1) for mod in TARGET_MODS])
    phase_bins, phase_min, phase_max = exact_bins(combined_P, NUM_BINS)
    phase_centers = 0.5 * (phase_bins[:-1] + phase_bins[1:])
    phase_thresholds = phase_bins[:-1]

    fig, ax = plt.subplots(figsize=(9, 5.5))
    for mod in TARGET_MODS:
        values = phase_raw[(mod, snr)].reshape(-1)
        counts, _ = np.histogram(values, bins=phase_bins)

        ax.step(phase_centers, counts, where="mid", color=COLORS[mod], linewidth=2.4, label=mod)
        ax.scatter(phase_centers, counts, color=COLORS[mod], s=18)

        for bin_id, (left, right, center, count) in enumerate(zip(phase_bins[:-1], phase_bins[1:], phase_centers, counts), start=1):
            phase_hist_rows.append({
                "snr_db": snr,
                "modulation": mod,
                "bin_id": bin_id,
                "bin_left": float(left),
                "bin_right": float(right),
                "bin_center": float(center),
                "count": int(count),
            })

        for row in crossing_counts(values, phase_thresholds, direction=">="):
            row.update({"snr_db": snr, "modulation": mod, "feature": "raw_phase", "direction": ">="})
            phase_cross_rows.append(row)

    ax.set_title(f"All-signal raw phase histogram, exact 20 bins, SNR = {snr} dB")
    ax.set_xlabel("Raw phase phase(t) = atan2(Q(t), I(t)), radians")
    ax.set_ylabel("Number of time samples")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    out_path = FIG_DIR / f"{safe_snr_name(snr)}_all_signals_raw_phase_exact20bins.png"
    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")

    fig, ax = plt.subplots(figsize=(9, 5.5))
    for mod in TARGET_MODS:
        df = pd.DataFrame([r for r in phase_cross_rows if r["snr_db"] == snr and r["modulation"] == mod])
        ax.plot(df["threshold"], df["crossing_count"], color=COLORS[mod], linewidth=2.4, marker="o", label=mod)

    ax.set_title(f"Raw phase threshold crossings, SNR = {snr} dB")
    ax.set_xlabel("Raw phase threshold value, radians")
    ax.set_ylabel("Number of time samples with phase(t) >= threshold")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    out_path = FIG_DIR / f"{safe_snr_name(snr)}_raw_phase_crossing_counts.png"
    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")

amp_hist_df = pd.DataFrame(amp_hist_rows)
phase_hist_df = pd.DataFrame(phase_hist_rows)
amp_cross_df = pd.DataFrame(amp_cross_rows)
phase_cross_df = pd.DataFrame(phase_cross_rows)
outer_df = pd.DataFrame(outer_rows)

amp_hist_csv = RESULT_DIR / "all_signals_raw_amplitude_exact20bin_histogram_counts.csv"
phase_hist_csv = RESULT_DIR / "all_signals_raw_phase_exact20bin_histogram_counts.csv"
amp_cross_csv = RESULT_DIR / "all_signals_raw_amplitude_threshold_crossing_counts.csv"
phase_cross_csv = RESULT_DIR / "all_signals_raw_phase_threshold_crossing_counts.csv"
outer_csv = RESULT_DIR / "raw_amplitude_highest_bin_outer_region_counts.csv"

amp_hist_df.to_csv(amp_hist_csv, index=False)
phase_hist_df.to_csv(phase_hist_csv, index=False)
amp_cross_df.to_csv(amp_cross_csv, index=False)
phase_cross_df.to_csv(phase_cross_csv, index=False)
outer_df.to_csv(outer_csv, index=False)

print(f"Saved: {amp_hist_csv}")
print(f"Saved: {phase_hist_csv}")
print(f"Saved: {amp_cross_csv}")
print(f"Saved: {phase_cross_csv}")
print(f"Saved: {outer_csv}")
print("\nHighest-amplitude-bin crossing summary:")
display(outer_df)

In [ ]:
# CELL 5: Combined summary plots for all SNRs
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.ravel()

for ax, snr in zip(axes, TARGET_SNRS):
    for mod in TARGET_MODS:
        df = amp_cross_df[(amp_cross_df["snr_db"] == snr) & (amp_cross_df["modulation"] == mod)]
        ax.plot(df["threshold"], df["crossing_count"], color=COLORS[mod], linewidth=2.0, marker="o", label=mod)
    ax.set_title(f"SNR = {snr} dB")
    ax.set_xlabel("Raw amplitude threshold")
    ax.set_ylabel("Count with A(t) >= threshold")
    ax.grid(True, alpha=0.3)
    ax.legend()

fig.suptitle("QAM16 vs QAM64 raw amplitude threshold crossings", fontsize=16, y=1.02)
fig.tight_layout()
combined_amp_cross_path = FIG_DIR / "all_target_snrs_raw_amplitude_crossing_counts.png"
fig.savefig(combined_amp_cross_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {combined_amp_cross_path}")
display(Image(filename=str(combined_amp_cross_path)))

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.ravel()

for ax, snr in zip(axes, TARGET_SNRS):
    combined_A = np.concatenate([amp_raw[(mod, snr)].reshape(-1) for mod in TARGET_MODS])
    bins, _, _ = exact_bins(combined_A, NUM_BINS)
    centers = 0.5 * (bins[:-1] + bins[1:])
    for mod in TARGET_MODS:
        counts, _ = np.histogram(amp_raw[(mod, snr)].reshape(-1), bins=bins)
        ax.step(centers, counts, where="mid", color=COLORS[mod], linewidth=2.0, label=mod)
    ax.axvspan(bins[-2], bins[-1], color="red", alpha=0.08)
    ax.set_title(f"SNR = {snr} dB")
    ax.set_xlabel("Raw amplitude A(t)")
    ax.set_ylabel("Number of time samples")
    ax.grid(True, alpha=0.3)
    ax.legend()

fig.suptitle("QAM16 vs QAM64 raw amplitude histograms, exact 20 bins", fontsize=16, y=1.02)
fig.tight_layout()
combined_amp_hist_path = FIG_DIR / "all_target_snrs_raw_amplitude_exact20bin_histograms.png"
fig.savefig(combined_amp_hist_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {combined_amp_hist_path}")
display(Image(filename=str(combined_amp_hist_path)))

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.ravel()

for ax, snr in zip(axes, TARGET_SNRS):
    for mod in TARGET_MODS:
        df = phase_cross_df[(phase_cross_df["snr_db"] == snr) & (phase_cross_df["modulation"] == mod)]
        ax.plot(df["threshold"], df["crossing_count"], color=COLORS[mod], linewidth=2.0, marker="o", label=mod)
    ax.set_title(f"SNR = {snr} dB")
    ax.set_xlabel("Raw phase threshold, radians")
    ax.set_ylabel("Count with phase(t) >= threshold")
    ax.grid(True, alpha=0.3)
    ax.legend()

fig.suptitle("QAM16 vs QAM64 raw phase threshold crossings", fontsize=16, y=1.02)
fig.tight_layout()
combined_phase_cross_path = FIG_DIR / "all_target_snrs_raw_phase_crossing_counts.png"
fig.savefig(combined_phase_cross_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {combined_phase_cross_path}")
display(Image(filename=str(combined_phase_cross_path)))

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes = axes.ravel()

for ax, snr in zip(axes, TARGET_SNRS):
    combined_P = np.concatenate([phase_raw[(mod, snr)].reshape(-1) for mod in TARGET_MODS])
    bins, _, _ = exact_bins(combined_P, NUM_BINS)
    centers = 0.5 * (bins[:-1] + bins[1:])
    for mod in TARGET_MODS:
        counts, _ = np.histogram(phase_raw[(mod, snr)].reshape(-1), bins=bins)
        ax.step(centers, counts, where="mid", color=COLORS[mod], linewidth=2.0, label=mod)
    ax.set_title(f"SNR = {snr} dB")
    ax.set_xlabel("Raw phase phase(t), radians")
    ax.set_ylabel("Number of time samples")
    ax.grid(True, alpha=0.3)
    ax.legend()

fig.suptitle("QAM16 vs QAM64 raw phase histograms, exact 20 bins", fontsize=16, y=1.02)
fig.tight_layout()
combined_phase_hist_path = FIG_DIR / "all_target_snrs_raw_phase_exact20bin_histograms.png"
fig.savefig(combined_phase_hist_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {combined_phase_hist_path}")
display(Image(filename=str(combined_phase_hist_path)))

In [ ]:
# CELL 6: Write README for exported results
readme_text = f"""
QAM16 vs QAM64 raw crossing analysis

Target SNR values: {TARGET_SNRS} dB
Bins per histogram: {NUM_BINS}
Signals used: all available signals for each modulation/SNR block

Processing rules:
- No random sample selection.
- No normalization.
- No mean subtraction.
- Raw amplitude: A(t) = sqrt(I(t)^2 + Q(t)^2).
- Raw phase: phase(t) = atan2(Q(t), I(t)), in radians.
- For each SNR, QAM16 and QAM64 share the exact combined min-to-max range split into 20 equal bins.
- Crossing counts report the number of raw time samples greater than or equal to each threshold.

Main figures:
- figures/*_all_signals_raw_amplitude_exact20bins.png
- figures/*_raw_amplitude_crossing_counts.png
- figures/*_all_signals_raw_phase_exact20bins.png
- figures/*_raw_phase_crossing_counts.png
- figures/all_target_snrs_raw_amplitude_crossing_counts.png
- figures/all_target_snrs_raw_amplitude_exact20bin_histograms.png
- figures/all_target_snrs_raw_phase_crossing_counts.png
- figures/all_target_snrs_raw_phase_exact20bin_histograms.png

Main tables:
- results/raw_amplitude_phase_summary_all_signals.csv
- results/all_signals_raw_amplitude_exact20bin_histogram_counts.csv
- results/all_signals_raw_phase_exact20bin_histogram_counts.csv
- results/all_signals_raw_amplitude_threshold_crossing_counts.csv
- results/all_signals_raw_phase_threshold_crossing_counts.csv
- results/raw_amplitude_highest_bin_outer_region_counts.csv

The highest-amplitude-bin table is the direct outer-region count summary: it shows how many raw time samples fall in the largest amplitude bucket for QAM16 and QAM64 at each SNR.
""".strip()

readme_path = EXP_DIR / "README_raw_crossing_analysis.txt"
readme_path.write_text(readme_text)
print(f"Saved: {readme_path}")

In [ ]:
# CELL 7: Zip all figures and tables for download
stamp = datetime.now().strftime("%Y%m%d_%H%M")
zip_base = Path("/kaggle/working") / f"qam16_qam64_raw_crossing_analysis_{stamp}"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=str(EXP_DIR))

print("Export complete")
print(f"Zip file: {zip_path}")
print("\nFiles included:")
for path in sorted(EXP_DIR.rglob("*")):
    if path.is_file():
        print(" -", path.relative_to(EXP_DIR))

try:
    from IPython.display import FileLink
    display(FileLink(zip_path))
except Exception:
    pass